# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing a FAIR^2 structured dataset using the `mlcroissant` library. You will learn how to reference all dataset components by their unique `@id`, clarify the schema, extract records, and perform initial analyses.

### Dataset Source
The dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

The dataset covers clinical, laboratory, imaging, and genetic data for three female patients with non-classical 21-hydroxylase deficiency (NC-21OHD) and infertility.

In [ ]:
# Install mlcroissant to enable schema-driven data loading
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. All metadata and subsequent references use the `@id` identifiers as defined by the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print metadata overview
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset @id: {metadata.id}")
print(f"RecordSets: {len(metadata.record_sets())} total")

## 2. Data Overview

Review available record sets, their unique `@id`s, and constituent fields (columns).

In Croissant, each record set, field, and column is uniquely referenced by its `@id`. Below, we enumerate all record sets, their fields, and corresponding `@id`s to facilitate further extraction.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.record_sets()
overview = {}
for rset in record_sets:
    print(f"\nRecordSet name: {rset.name}, @id: {rset.id}")
    field_ids = [field.id for field in rset.fields]
    print("Fields by @id:")
    for field in rset.fields:
        print(f"  - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    overview[rset.id] = field_ids

# Show example records from each record set
for rset in record_sets:
    print(f"\nSample records from RecordSet @id: {rset.id}")
    records_iter = dataset.records(record_set=rset.id)
    for i, record in enumerate(records_iter):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction

Load data from each record set into DataFrames for analysis.
Each record set and field is referenced strictly by its `@id`, as shown above.
Below, all defined record sets are loaded; you can select specific sets for deeper analysis using their `@id`s.

In [ ]:
# Extract and store each record set into a DataFrame
record_set_ids = [rset.id for rset in dataset.metadata.record_sets()]
dataframes = {}

for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df

# Print columns for each loaded DataFrame
for rset_id in dataframes:
    print(f"Columns in RecordSet @id: {rset_id}")
    print(dataframes[rset_id].columns.tolist())
    display(dataframes[rset_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply data transformations and basic filtering to prepare the dataset for visualization and further analysis.

Below, numeric columns are identified by their field `@id`, outliers are filtered, values normalized, and grouping demonstrated (where possible). All column references use the Croissant `@id`.

In [ ]:
# Choose an example RecordSet for EDA
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
else:
    raise ValueError("No record sets found in metadata.")

# Identify numeric fields by @id in this RecordSet
numeric_fields = []
for field in dataset.metadata.record_set(selected_record_set_id).fields:
    if field.data_type in ['Integer', 'Float', 'Number']:
        numeric_fields.append(field.id)

print(f"Numeric fields in RecordSet @id {selected_record_set_id}: {numeric_fields}")

# If any numeric field is available, perform filtering/EDA
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    df = dataframes[selected_record_set_id]

    # Drop NA for the analysis
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = numeric_series.mean()  # Example threshold: mean
    filtered_df = df[numeric_series > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series[filtered_df.index] - numeric_series.mean()) / numeric_series.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical column
    cat_fields = [field.id for field in dataset.metadata.record_set(selected_record_set_id).fields if field.data_type == 'Text']
    if cat_fields:
        group_field_id = cat_fields[0]
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
else:
    print("No numeric fields detected for EDA; please check dataset schema for quantitative columns.")

## 5. Visualization

Visualize the distribution of a numeric field, or relationships between two fields, using matplotlib or pandas built-in plotting.

Column references remain strictly by their `@id`. Adapt the plot for your analytic needs (histogram, boxplot, scatter, etc).

In [ ]:
import matplotlib.pyplot as plt

# Example visualization: if numeric fields present, plot histogram
if numeric_fields:
    plt.figure(figsize=(6,4))
    numeric_series.dropna().plot.hist(bins=10, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id} (RecordSet @id: {selected_record_set_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot
    plt.figure(figsize=(4,6))
    numeric_series.dropna().plot.box()
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("No numeric fields detected for visualization.")

## 6. Conclusion

This notebook provides a template for exploring a FAIR^2 structured dataset using the Croissant schema and `mlcroissant`.

- **Record sets, fields, and columns are referenced strictly by their `@id`.**
- **Demonstrated how to load raw tabular data, normalize, filter, group, and visualize.**
- **For further analysis, carefully review field types and their corresponding `@id` in the data schema.**

The dataset is limited in size and scope (three patients), but illustrates a framework for reproducible, schema-aware biomedical data exploration. Please refer to the Croissant metadata for more precise field definitions and adapt processing to specific research requirements.